<a href="https://colab.research.google.com/github/ASpacesys/camera-pose-estimation-from-landmarks/blob/main/Camera_Pose_Estimation_from_Landmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding an Observer's Position from a Photograph: An Exercise in Perspective Geometry

This notebook aims to find the exact position of any photo given identifiable landmarks with known coordinates. It is split into two sections: a fast, rough search which narrows the range of possible locations to a single line, and a more detailed search which attempts to find the exact latitude, longitude, and altitude of the observer.

**For reliable results:**
- Use **5 ≤ N ≤ 8** landmarks for searches. Lower values won't yield correct solutions, but higher values are prone to high pixel uncertainty.
- Try to pick points that span a large depth field and field of view. The more spread out the points, the more accurate the detailed search will be.

**Notes:**
 -
 - As of now, the line of possible locations uses cv2's solvePNP function. It may be replaced in the future with a method which doesn't require camera intrinsics, although the current version is relatively accurate.
   - Known issue: The line of possible locations can be flipped along the Z-axis of the ENU origin point.  
 - The gradient descent algorithm for a more exact location may be incorrect. This is a flaw in the methodology which will eventually be fixed. However, the line of possible locations is accurate to around 200 feet per mile from the ENU origin.
 - The earth's curvature is also unaccounted for, although this error is much less significant than the above. ENU coordinates may be replaced with ECEF and geodetics in the future.

In [ ]:
#@title Install Dependencies

!pip install pymap3d opencv-python -q

In [ ]:
#@title Enter Google Maps API Key

# Used for interactive world coordinate entry and Google Maps Visualization. You can get a free personal Maps Demo API Key.
GOOGLE_MAPS_API_KEY = ""

In [ ]:
#@title Input Pixel Coordinates
import sys
IN_COLAB = 'google.colab' in sys.modules

if not IN_COLAB:
    print("Use manual entry instead, or run this notebook in Colab.")
else:
    from google.colab import files, output
    from PIL import Image
    from IPython.display import HTML, display
    import io, base64

    print("Choose your image file:")
    uploaded = files.upload()
    _fname = next(iter(uploaded))
    _img_bytes = uploaded[_fname]
    _img = Image.open(io.BytesIO(_img_bytes)).convert('RGB')
    image_width_px, image_height_px = _img.size
    print(f"Loaded '{_fname}': {image_width_px} x {image_height_px} px")

    _buf = io.BytesIO()
    _img.save(_buf, format='JPEG', quality=90)
    _img_b64 = base64.b64encode(_buf.getvalue()).decode('utf-8')

    image_points_px = []

    def set_image_points(points):
        global image_points_px
        image_points_px = points
        print(f"Received {len(points)} image point(s) from the widget.")

    output.register_callback('notebook.set_image_points', set_image_points)

    MAX_DISPLAY_WIDTH = 1000
    _scale = min(1.0, MAX_DISPLAY_WIDTH / image_width_px)
    _disp_w = int(image_width_px * _scale)
    _disp_h = int(image_height_px * _scale)

    _html = '''
<div>
  <div style="margin-bottom:6px;">
    <button id="imgUndoBtn">Undo last point</button>
    <button id="imgClearBtn">Clear all</button>
    <button id="imgDoneBtn" style="font-weight:bold;">Done \u2014 send points to Python</button>
    <span id="imgCountLabel" style="margin-left:10px;"></span>
  </div>
  <canvas id="imgCanvas" width="__DISP_W__" height="__DISP_H__" style="border:1px solid #ccc; cursor:crosshair;"></canvas>
</div>
<script>
(function() {
  const scale = __SCALE__;
  const canvas = document.getElementById('imgCanvas');
  const ctx = canvas.getContext('2d');
  const img = new Image();
  let points = [];

  function redraw() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
    points.forEach((p, i) => {
      const cx = p[0]*scale, cy = p[1]*scale;
      ctx.beginPath();
      ctx.arc(cx, cy, 6, 0, 2*Math.PI);
      ctx.fillStyle = 'red';
      ctx.fill();
      ctx.fillStyle = 'white';
      ctx.font = 'bold 14px sans-serif';
      ctx.fillText(String(i+1), cx+8, cy-8);
    });
    document.getElementById('imgCountLabel').innerText = points.length + ' point(s) clicked';
  }

  img.onload = () => { redraw(); };
  img.src = 'data:image/jpeg;base64,__IMG_B64__';

  canvas.addEventListener('click', (e) => {
    const rect = canvas.getBoundingClientRect();
    const cx = e.clientX - rect.left;
    const cy = e.clientY - rect.top;
    points.push([cx/scale, cy/scale]);
    redraw();
  });

  document.getElementById('imgUndoBtn').onclick = () => { points.pop(); redraw(); };
  document.getElementById('imgClearBtn').onclick = () => { points = []; redraw(); };
  document.getElementById('imgDoneBtn').onclick = () => {
    google.colab.kernel.invokeFunction('notebook.set_image_points', [points], {});
    document.getElementById('imgCountLabel').innerText = points.length + ' point(s) sent to Python!';
  };
})();
</script>
'''
    _html = (_html.replace('__DISP_W__', str(_disp_w))
                   .replace('__DISP_H__', str(_disp_h))
                   .replace('__SCALE__', str(_scale))
                   .replace('__IMG_B64__', _img_b64))
    display(HTML(_html))

After clicking **Done** above, run this cell to build the pixel coordinate array.

In [ ]:
#@title Build Pixel Coordinate Array
import numpy as np
if not IN_COLAB or not image_points_px:
    print("Unable to use interactive coordinates.")
else:
    pixel_coords = np.array(image_points_px, dtype=np.float64)
    # Assume optical center is the center of the image
    cx, cy = image_width_px / 2, image_height_px / 2

    # Random guess for focal length
    fx = fx if 'fx' in globals() else image_width_px
    fy = fy if 'fy' in globals() else image_width_px

    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.zeros(5, dtype=np.float64)

    print(pixel_coords)

# Interactive World Coordinate Entry
Instead of finding latitude, longitude, and altitude through other resources, the next two cells give the option to click landmarks on Google Maps. This is most useful for aerial images. For ground or near-ground images, skip ahead to input coordinates manually.

In [ ]:
#@title Input World Coordinates (requires API key)

import sys
IN_COLAB = 'google.colab' in sys.modules

map_center_lat, map_center_lon = 37.70819703505958, -122.29786392135888
map_zoom = 11

if not IN_COLAB:
    print("This interactive widget requires Google Colab.")
else:
    from google.colab import output
    from IPython.display import HTML, display

    map_points_latlng = []

    def set_map_points(points):
        global map_points_latlng
        map_points_latlng = points
        print(f"Received {len(points)} map point(s) from the widget.")

    output.register_callback('notebook.set_map_points', set_map_points)

    _html = '''
<div>
  <div style="margin-bottom:6px;">
    <button id="mapUndoBtn">Undo last point</button>
    <button id="mapClearBtn">Clear all</button>
    <button id="mapDoneBtn" style="font-weight:bold;">Done \u2014 send points to Python</button>
    <span id="mapCountLabel" style="margin-left:10px;"></span>
  </div>
  <div id="pnpMap" style="width:100%; height:600px; border:1px solid #ccc;"></div>
</div>
<script>
(function() {
  let clickedPoints = [];
  let markers = [];
  let map;

  function relabel() {
    markers.forEach((m, i) => m.setLabel(String(i+1)));
    document.getElementById('mapCountLabel').innerText = clickedPoints.length + ' point(s) clicked';
  }

  window.pnpInitMap = function() {
    map = new google.maps.Map(document.getElementById('pnpMap'), {
      center: { lat: __LAT__, lng: __LON__ },
      zoom: __ZOOM__,
      mapTypeId: 'hybrid'
    });
    map.addListener('click', (e) => {
      const lat = e.latLng.lat();
      const lng = e.latLng.lng();
      clickedPoints.push({lat: lat, lng: lng});
      const marker = new google.maps.Marker({
        position: {lat: lat, lng: lng},
        map: map,
        label: String(clickedPoints.length)
      });
      markers.push(marker);
      relabel();
    });
  };

  document.getElementById('mapUndoBtn').onclick = () => {
    clickedPoints.pop();
    const m = markers.pop();
    if (m) m.setMap(null);
    relabel();
  };
  document.getElementById('mapClearBtn').onclick = () => {
    clickedPoints = [];
    markers.forEach(m => m.setMap(null));
    markers = [];
    relabel();
  };
  document.getElementById('mapDoneBtn').onclick = () => {
    google.colab.kernel.invokeFunction('notebook.set_map_points', [clickedPoints], {});
    document.getElementById('mapCountLabel').innerText = clickedPoints.length + ' point(s) sent to Python!';
  };
})();
</script>
<script src="https://maps.googleapis.com/maps/api/js?key=__KEY__&callback=pnpInitMap" async defer></script>
'''
    _html = (_html.replace('__LAT__', str(map_center_lat))
                   .replace('__LON__', str(map_center_lon))
                   .replace('__ZOOM__', str(map_zoom))
                   .replace('__KEY__', GOOGLE_MAPS_API_KEY))
    display(HTML(_html))

After clicking **Done** above, run this cell to build the world coordinate array.

In [ ]:
#@title Build World Coordinates Array
import requests
import numpy as np
import pymap3d as pm

if not IN_COLAB or not map_points_latlng:
    print("Either no map points were captured or Google Colab is not running.")
else:
    def get_msl_elevations(latlng_list):
        lats = ",".join(str(p['lat']) for p in latlng_list)
        lngs = ",".join(str(p['lng']) for p in latlng_list)

        url = f"https://api.open-meteo.com/v1/elevation?latitude={lats}&longitude={lngs}"
        resp = requests.get(url).json()

        if 'elevation' not in resp:
            raise RuntimeError(f"Open-Meteo API error: {resp}")
        return resp['elevation']

    elevations_msl = get_msl_elevations(map_points_latlng)
    landmarks_lla = [(p['lat'], p['lng'], elev) for p, elev in zip(map_points_latlng, elevations_msl)]

    if 'landmarks_lla' in globals() and len(pixel_coords) != len(landmarks_lla):
        print(f"\n\u26a0\ufe0f  MISMATCH: {len(pixel_coords)} image points vs {len(landmarks_lla)} landmarks. "
              "Make sure you clicked the same number of points, in the same order, on both widgets.")
    else:
        print(f"\nOK -- {len(pixel_coords)} points match landmarks_lla count.")
        print(landmarks_lla)
        print(f"Built landmarks_lla from {len(landmarks_lla)} clicked map points (altitude = MSL, via Elevation API):")
        for i, (la, lo, al) in enumerate(landmarks_lla):
          print(f"  {i+1}: {la:.6f}, {lo:.6f}, {al:.1f} m MSL")

        lat0, lon0, alt0 = landmarks_lla[0]
        world_points = np.array([
        pm.geodetic2enu(lat, lon, alt, lat0, lon0, alt0)
        for lat, lon, alt in landmarks_lla
        ], dtype=np.float64)


# Manual World Coordinate Entry
For ground or near-ground images, world coordinates should be input manually. This allows you to use points such as faces of buildings as landmarks. Points should be input in the format of (latitude, longitude, altitude (MSL, meters)).

In [ ]:
#@title Manual World Coordinate Input
import numpy as np
import pymap3d as pm

# Add points in the format (latitude, longitude, altitude (MSL, in meters))
landmarks_lla = [
(42.28419675439698, -83.74130034799083, 270),
(42.284242890354214, -83.74120311791316, 270),
(42.28428654585253, -83.74129699522953, 270),
(42.28591084824175, -83.74115475468355, 268),
(42.28597781795939, -83.74113463811577, 268)
]

if 'landmarks_lla' in globals() and len(pixel_coords) != len(landmarks_lla):
    print(f"\n\u26a0\ufe0f  MISMATCH: {len(pixel_coords)} image points vs {len(landmarks_lla)} landmarks. "
          "Make sure you clicked the same number of points, in the same order, on both widgets.")
else:
    print(f"\nOK -- {len(pixel_coords)} points match landmarks_lla count.")
    print(f"Built landmarks_lla from {len(landmarks_lla)} clicked map points (altitude = MSL, via Elevation API):")
    for i, (la, lo, al) in enumerate(landmarks_lla):
      print(f"  {i+1}: {la:.6f}, {lo:.6f}, {al:.1f} m MSL")

    lat0, lon0, alt0 = landmarks_lla[0]
    world_points = np.array([
    pm.geodetic2enu(lat, lon, alt, lat0, lon0, alt0)
    for lat, lon, alt in landmarks_lla
    ], dtype=np.float64)

# Solving for possible positions
Zooming in while moving backwards or zooming out while moving forwards ("dolly zooms") can make objects in the frame appear the same size. While this makes identifying the exact location harder, it is easy to determine the line in 3D space which the observer might lie on. Note that this line does not correspond to the direction in which the observer is looking.

In [ ]:
#@title Find line of possible observer positions
from scipy.optimize import brentq
from scipy.constants import foot
import cv2
from scipy.optimize import minimize_scalar

def solve_observer_pose_from_pixels(world_points, pixel_coords, K, dist_coeffs=None):
    """Pose-only solve for a GIVEN (fixed) camera intrinsics matrix K.
    Used directly when you know your focal length, and used internally by
    Section 6's focal-length sweep (which needs to solve pose at many different f values)."""
    world_points = np.asarray(world_points, dtype=np.float64)
    pixel_coords = np.asarray(pixel_coords, dtype=np.float64)
    if dist_coeffs is None:
        dist_coeffs = np.zeros(5, dtype=np.float64)
    n = len(world_points)
    if n < 3:
        raise ValueError("Need at least 3 landmarks.")
    flag = cv2.SOLVEPNP_ITERATIVE if n >= 4 else cv2.SOLVEPNP_P3P
    ok, rvec, tvec = cv2.solvePnP(world_points, pixel_coords, K, dist_coeffs, flags=flag)
    if not ok:
        raise RuntimeError("solvePnP failed to find a solution.")
    if n >= 4:
        rvec, tvec = cv2.solvePnPRefineLM(world_points, pixel_coords, K, dist_coeffs, rvec, tvec)
    R_world_to_local, _ = cv2.Rodrigues(rvec)
    E = (-R_world_to_local.T @ tvec).flatten()
    return E, R_world_to_local.T

def focal_length_ambiguity_line(world_points, pixel_coords, cx, cy, dist_coeffs,
                                 f_nominal, f_uncertainty_frac=0.15, n_samples=25, fy_over_fx=1.0):
    """
    Sweeps fx over [f_nominal*(1-frac), f_nominal*(1+frac)] (your plausible uncertainty range),
    re-solving the observer's position at each value, then fits the best-fit 3D line
    (via PCA / total least squares) through the resulting ENU positions.

    Returns:
      E0             : (3,) point on the line -- the position at f_nominal (your best guess)
      direction      : (3,) unit direction vector of the line, in ENU
                       (physically close to your camera's viewing direction toward the scene)
      t_range        : (t_min, t_max) meters along the line spanned by your f-uncertainty range
      r2             : fraction of variance in the swept positions explained by the line
                       (close to 1.0 = the ambiguity really is ~1D; this line is a good summary)
      fit_residual_m : RMS perpendicular distance of the samples from the fitted line (meters)
    """
    if dist_coeffs is None:
        dist_coeffs = np.zeros(5, dtype=np.float64)

    f_values = np.linspace(f_nominal * (1 - f_uncertainty_frac),
                            f_nominal * (1 + f_uncertainty_frac), n_samples)
    positions = []
    for f in f_values:
        K_test = np.array([[f, 0, cx], [0, f * fy_over_fx, cy], [0, 0, 1]], dtype=np.float64)
        E, _ = solve_observer_pose_from_pixels(world_points, pixel_coords, K_test, dist_coeffs)
        positions.append(E)
    positions = np.array(positions)

    centroid = positions.mean(axis=0)
    _, s, vt = np.linalg.svd(positions - centroid)
    direction = vt[0]
    r2 = s[0]**2 / np.sum(s**2)
    # orient direction so increasing t corresponds to increasing f, for intuitive t_range
    if np.dot(positions[-1] - positions[0], direction) < 0:
        direction = -direction

    # anchor point: position at exactly f_nominal (your actual best-guess estimate)
    K_nominal = np.array([[f_nominal, 0, cx], [0, f_nominal * fy_over_fx, cy], [0, 0, 1]], dtype=np.float64)
    E0, _ = solve_observer_pose_from_pixels(world_points, pixel_coords, K_nominal, dist_coeffs)

    t_vals = (positions - E0) @ direction
    perp_dist = np.linalg.norm((positions - E0) - np.outer(t_vals, direction), axis=1)
    fit_residual_m = np.sqrt(np.mean(perp_dist**2))

    return E0, direction, (t_vals.min(), t_vals.max()), r2, fit_residual_m


f_uncertainty_frac = 0.15

E0, direction, t_range, r2, fit_residual_m = focal_length_ambiguity_line(
    world_points, pixel_coords, cx, cy, dist_coeffs, f_nominal=fx, f_uncertainty_frac=f_uncertainty_frac
)

print(f"Line-fit quality: R^2 = {r2:.4f}  (RMS perpendicular residual = {fit_residual_m:.3f} m)")
print()
print("Line of possible observer positions, in ENU meters (relative to landmark 1):")
print(f"  Point on line   E0 = ({E0[0]:.3f}, {E0[1]:.3f}, {E0[2]:.3f})")
print(f"  Direction (unit) d = ({direction[0]:.5f}, {direction[1]:.5f}, {direction[2]:.5f})")
print()
print("Parametric equation:")
print(f"  East(t)  = {E0[0]:.3f} + t * ({direction[0]:.5f})")
print(f"  North(t) = {E0[1]:.3f} + t * ({direction[1]:.5f})")
print(f"  Up(t)    = {E0[2]:.3f} + t * ({direction[2]:.5f})")

def altitude_at_t(t):
    pt = E0 + t * direction
    return pm.enu2geodetic(pt[0], pt[1], pt[2], lat0, lon0, alt0)[2]

def find_t_for_altitude_m(target_alt_m, t_search_radius=1e9):
    """Finds t such that altitude_at_t(t) == target_alt_m, via bracketed root-finding."""
    f = lambda t: altitude_at_t(t) - target_alt_m
    lo, hi = -1.0, 1.0
    while f(lo) * f(hi) > 0:
        lo *= 2
        hi *= 2
        if abs(lo) > t_search_radius:
            raise RuntimeError(
                f"Couldn't find a t where the line reaches {target_alt_m:.1f} m altitude "
                "within the search radius -- the line may be nearly horizontal (direction[2] ~ 0)."
            )
    return brentq(f, lo, hi)

# --- EDIT ME: altitude range to display, in feet ---
alt_min_ft, alt_max_ft = -20000.0, 20000.0

t_alt_min = find_t_for_altitude_m(alt_min_ft * foot)
t_alt_max = find_t_for_altitude_m(alt_max_ft * foot)
# order so t_alt_min < t_alt_max regardless of which direction altitude increases along the line
t_alt_min, t_alt_max = sorted([t_alt_min, t_alt_max])

print(f"t = {t_alt_min:.3f} -> altitude {altitude_at_t(t_alt_min)/foot:.0f} ft")
print(f"t = {t_alt_max:.3f} -> altitude {altitude_at_t(t_alt_max)/foot:.0f} ft")

# Solving for an exact position
In order to attempt to find the exact position, gradient descent is used to identify the most likely observation point. This algorithm takes around 90 seconds to run in total, and the output is not as reliable as the line of possible positions computed from the previous cell.

It is not recommended to use this exact position for near-ground images, as other factors such as nearby building height can be used along with the more accurate line of possible positions to pinpoint locations more precisely.

In [ ]:
import torch
import itertools

# ---------------------------------------------------------
# Use P and pixel_coords already defined earlier in the notebook
# (P from pm.geodetic2enu, pixel_coords from your pixel array)
# ---------------------------------------------------------
P_t = torch.as_tensor(world_points, dtype=torch.float64)
pixel_coords_t = torch.as_tensor(pixel_coords, dtype=torch.float64)  # already (N, 2), no need to slice

N = P_t.shape[0]
EPS = 1e-8
scene_scale = P_t.norm(dim=1).max().item()

# ---------------------------------------------------------
# Camera basis / projection (with principal point cx, cy)
# ---------------------------------------------------------
def build_basis(n_raw, u_raw):
    n = n_raw / (n_raw.norm() + EPS)
    u_proj = u_raw - (u_raw @ n) * n
    u = u_proj / (u_proj.norm() + EPS)
    r = torch.linalg.cross(n, u)
    return n, u, r

def project(points, O, n, u, r, f, cx, cy):
    vec = points - O
    denom = vec @ n
    denom = torch.where(denom.abs() < EPS, denom + EPS, denom)
    x = (vec @ r) / denom * f + cx
    y = (vec @ u) / denom * f + cy
    return torch.stack([x, y], dim=1)

def reproj_loss(O, n_raw, u_raw, f_raw, cx, cy):
    n, u, r = build_basis(n_raw, u_raw)
    f = torch.nn.functional.softplus(f_raw)
    proj = project(P_t, O, n, u, r, f, cx, cy)
    return torch.mean((proj - pixel_coords_t) ** 2)

# ---------------------------------------------------------
# Single optimization run (LBFGS) from a given init
# ---------------------------------------------------------
def run_lbfgs(O0, n0, u0, f0, cx0, cy0, max_iter=200):
    O = O0.clone().requires_grad_(True)
    n_raw = n0.clone().requires_grad_(True)
    u_raw = u0.clone().requires_grad_(True)
    f_raw = f0.clone().requires_grad_(True)
    cx = cx0.clone().requires_grad_(True)
    cy = cy0.clone().requires_grad_(True)

    optimizer = torch.optim.LBFGS([O, n_raw, u_raw, f_raw, cx, cy], lr=1.0,
                                   max_iter=max_iter, line_search_fn='strong_wolfe')

    def closure():
        optimizer.zero_grad()
        loss = reproj_loss(O, n_raw, u_raw, f_raw, cx, cy)
        loss.backward()
        return loss

    optimizer.step(closure)
    final_loss = reproj_loss(O, n_raw, u_raw, f_raw, cx, cy).item()
    return final_loss, O.detach(), n_raw.detach(), u_raw.detach(), f_raw.detach(), cx.detach(), cy.detach()

# ---------------------------------------------------------
# Polish: repeated LBFGS rounds from a given starting point
# ---------------------------------------------------------
def polish(O0, n0, u0, f0, cx0, cy0, n_rounds=10, max_iter=300):
    O = O0.clone().requires_grad_(True)
    n_raw = n0.clone().requires_grad_(True)
    u_raw = u0.clone().requires_grad_(True)
    f_raw = f0.clone().requires_grad_(True)
    cx = cx0.clone().requires_grad_(True)
    cy = cy0.clone().requires_grad_(True)

    prev_loss = float('inf')
    for rnd in range(n_rounds):
        optimizer = torch.optim.LBFGS([O, n_raw, u_raw, f_raw, cx, cy], lr=1.0,
                                       max_iter=max_iter, max_eval=max_iter * 2,
                                       tolerance_grad=1e-14, tolerance_change=1e-16,
                                       line_search_fn='strong_wolfe')

        def closure():
            optimizer.zero_grad()
            loss = reproj_loss(O, n_raw, u_raw, f_raw, cx, cy)
            loss.backward()
            return loss

        optimizer.step(closure)
        cur_loss = reproj_loss(O, n_raw, u_raw, f_raw, cx, cy).item()

        if abs(prev_loss - cur_loss) < 1e-16:
            break
        prev_loss = cur_loss

    final_loss = reproj_loss(O, n_raw, u_raw, f_raw, cx, cy).item()
    return final_loss, O.detach(), n_raw.detach(), u_raw.detach(), f_raw.detach(), cx.detach(), cy.detach()

# ---------------------------------------------------------
# Multi-restart search (tries both n and -n directions to
# break the mirror ambiguity)
# ---------------------------------------------------------
torch.manual_seed(0)
results = []
n_restarts = 60

for trial in range(n_restarts):
    n0 = torch.randn(3, dtype=torch.float64)
    u0 = torch.randn(3, dtype=torch.float64)
    n0_unit = n0 / n0.norm()

    for sign in (+1.0, -1.0):
        O0 = sign * n0_unit * scene_scale * (0.5 + torch.rand(1).item() * 2.0)
        f0 = torch.tensor(500.0 + torch.rand(1).item() * 3000.0, dtype=torch.float64)
        cx0 = torch.zeros((), dtype=torch.float64)
        cy0 = torch.zeros((), dtype=torch.float64)

        try:
            loss, O_f, n_f, u_f, f_f, cx_f, cy_f = run_lbfgs(O0, n0, u0, f0, cx0, cy0)
        except Exception:
            continue

        results.append((loss, O_f, n_f, u_f, f_f, cx_f, cy_f))

results.sort(key=lambda r: r[0])
print("Top 5 raw restarts before polishing:")
for i, r in enumerate(results[:5]):
    print(f"  #{i}: loss={r[0]:.6f} | RMS={r[0]**0.5:.4f} px")

# ---------------------------------------------------------
# Polish top 5 candidates, keep the overall best
# ---------------------------------------------------------
print("\nPolishing top candidates...")
polished = []
for i, r in enumerate(results[:5]):
    _, O0, n0, u0, f0, cx0, cy0 = r
    loss, O_f, n_f, u_f, f_f, cx_f, cy_f = polish(O0, n0, u0, f0, cx0, cy0)
    print(f"  candidate #{i}: polished loss={loss:.8f} | RMS={loss**0.5:.6f} px")
    polished.append((loss, O_f, n_f, u_f, f_f, cx_f, cy_f))

polished.sort(key=lambda r: r[0])
best_loss, O_best, n_raw_best, u_raw_best, f_raw_best, cx_best, cy_best = polished[0]

with torch.no_grad():
    n_final, u_final, r_final = build_basis(n_raw_best, u_raw_best)
    f_final = torch.nn.functional.softplus(f_raw_best)
    proj_final = project(P_t, O_best, n_final, u_final, r_final, f_final, cx_best, cy_best)
    rms_final = torch.sqrt(torch.mean((proj_final - pixel_coords_t) ** 2)).item()


# 1. Use torch.norm() instead of .norm() on a numpy array
if torch.norm(O_best + n_final) > torch.norm(O_best):
    # 2. In-place multiplication modifies the original tensor's values
    n_final *= -1

E_est = O_best.numpy()
obs_lat, obs_lon, obs_alt = pm.enu2geodetic(E_est[0], E_est[1], E_est[2], lat0, lon0, alt0)


print("\n=== FINAL BEST RESULT ===")
print("O (observer position, ENU):", O_best.numpy())
print(f"O (observer location, lat/long/alt): {obs_lat:.6f}, {obs_lon:.6f}, alt {obs_alt:.1f} m")
print("n (gaze direction):        ", n_final.numpy())
print("u (up vector):             ", u_final.numpy())
print("r (right, = n x u):        ", r_final.numpy())
print("f (focal length, px):      ", f_final.item())
print("cx, cy (principal pt):     ", cx_best.item(), cy_best.item())
print("Final RMS reprojection error (px):", rms_final)

In [ ]:
#@title Adjust observer altitude if below median landmark altitude, and re-solve position
import numpy as np

# ---------------------------------------------------------
# Compare observer altitude (obs_alt, in meters) to the median
# of landmark altitudes (landmarks_lla altitudes are also in meters)
# ---------------------------------------------------------
landmark_altitudes_m = [alt for (lat, lon, alt) in landmarks_lla]
median_alt_m = np.median(landmark_altitudes_m)

first_landmark_alt_m = landmarks_lla[0][2]

if obs_alt < median_alt_m:
    # Reflect obs_alt across the first landmark's altitude (in meters):
    # new_alt = first_landmark_alt + (first_landmark_alt - obs_alt)
    updated_obs_alt_m = first_landmark_alt_m + (first_landmark_alt_m - obs_alt)
    print(f"obs_alt ({obs_alt:.2f} m) is below median landmark altitude ({median_alt_m:.2f} m); "
          f"reflecting across first landmark altitude ({first_landmark_alt_m:.2f} m) "
          f"-> updated_obs_alt = {updated_obs_alt_m:.2f} m")
else:
    updated_obs_alt_m = obs_alt
    print(f"obs_alt ({obs_alt:.2f} m) is at/above median landmark altitude ({median_alt_m:.2f} m); "
          f"using as-is -> updated_obs_alt = {updated_obs_alt_m:.2f} m")

# ---------------------------------------------------------
# Solve for t at the (possibly updated) altitude, then plug
# back into the parametric line equations to get ENU coordinates
# ---------------------------------------------------------
t_alt_new = find_t_for_altitude_m(updated_obs_alt_m)

pt_alt_enu = E0 + t_alt_new * direction
E_alt, N_alt, U_alt = pt_alt_enu

lat_alt, lon_alt, alt_alt_m = pm.enu2geodetic(E_alt, N_alt, U_alt, lat0, lon0, alt0)

print(f"\nt_alt_new = {t_alt_new:.6f}")
print(f"Solved ENU coordinates (altitude-based): East={E_alt:.3f}, North={N_alt:.3f}, Up={U_alt:.3f}")
print(f"Solved geodetic (altitude-based): lat={lat_alt:.8f}, lon={lon_alt:.8f}, alt={alt_alt_m:.2f} m "
      f"({alt_alt_m/foot:.1f} ft)")

# ---------------------------------------------------------
# obs_lat / obs_lon: no reflection logic needed here. Instead,
# extend an imaginary vertical line at (obs_lat, obs_lon) --
# i.e. fixed East/North, varying Up -- and find the point on the
# parametric line closest to that vertical line. Since the vertical
# line has constant East/North for all altitudes, this reduces to a
# 2D closest-point problem in the East-North plane only.
# ---------------------------------------------------------
# East/North of the vertical line (altitude doesn't matter here --
# any reference altitude gives the same East/North in this local
# tangent-plane approximation).
E_target, N_target, _ = pm.geodetic2enu(obs_lat, obs_lon, alt0, lat0, lon0, alt0)

E0_EN = np.array([E0[0], E0[1]])
dir_EN = np.array([direction[0], direction[1]])
target_EN = np.array([E_target, N_target])

dir_EN_normsq = np.dot(dir_EN, dir_EN)
if dir_EN_normsq < 1e-12:
    raise RuntimeError(
        "The parametric line's direction has ~zero East/North component "
        "(it's nearly vertical), so it doesn't meaningfully approach any "
        "vertical line at a fixed lat/lon -- closest-point-in-EN is undefined."
    )

t_latlon_new = np.dot(target_EN - E0_EN, dir_EN) / dir_EN_normsq

pt_latlon_enu = E0 + t_latlon_new * direction
E_ll, N_ll, U_ll = pt_latlon_enu

lat_ll, lon_ll, alt_ll_m = pm.enu2geodetic(E_ll, N_ll, U_ll, lat0, lon0, alt0)

print(f"\nt_latlon_new = {t_latlon_new:.6f}")
print(f"Solved ENU coordinates (lat/lon-based): East={E_ll:.3f}, North={N_ll:.3f}, Up={U_ll:.3f}")
print(f"Solved geodetic (lat/lon-based): lat={lat_ll:.8f}, lon={lon_ll:.8f}, alt={alt_ll_m:.2f} m "
      f"({alt_ll_m/foot:.1f} ft)")

# ---------------------------------------------------------
# Average the two solutions in ENU, then convert back to geodetic
# ---------------------------------------------------------
pt_avg_enu = (pt_alt_enu + pt_latlon_enu) / 2.0
E_avg, N_avg, U_avg = pt_avg_enu

lat_avg, lon_avg, alt_avg_m = pm.enu2geodetic(E_avg, N_avg, U_avg, lat0, lon0, alt0)

print(f"\nAveraged ENU coordinates: East={E_avg:.3f}, North={N_avg:.3f}, Up={U_avg:.3f}")
print(f"Averaged geodetic: lat={lat_avg:.8f}, lon={lon_avg:.8f}, alt={alt_avg_m:.2f} m "
      f"({alt_avg_m/foot:.1f} ft)")

In [ ]:
#@title Visualization 1: KML File (can be imported to Google Earth)
import numpy as np, pymap3d as pm

gray = np.array([0x88, 0x88, 0x88])
green = np.array([0x1a, 0x9e, 0x1a])

t_samples = np.linspace(t_alt_min, t_alt_max, 200)
line_pts = np.array([E0 + t * direction for t in t_samples])
lla = np.array([pm.enu2geodetic(p[0], p[1], p[2], lat0, lon0, alt0) for p in line_pts])
lats, lons, alts = lla[:, 0], lla[:, 1], lla[:, 2]
max_alt = max(alts.max(), 1e-6)

kml = ['<?xml version="1.0" encoding="UTF-8"?>',
       '<kml xmlns="http://www.opengis.net/kml/2.2"><Document>',
       '<name>Observer position line</name>']

for i in range(len(lats) - 1):
    frac = (alts[i] + alts[i+1]) / 2 / max_alt
    r, g, b = (gray + frac * (green - gray)).astype(int)
    color = f'ff{b:02x}{g:02x}{r:02x}'  # KML color format is aabbggrr, not rrggbb
    kml.append(f'''<Placemark>
  <Style><LineStyle><color>{color}</color><width>5</width></LineStyle></Style>
  <LineString>
    <altitudeMode>absolute</altitudeMode>
    <coordinates>{lons[i]:.7f},{lats[i]:.7f},{alts[i]:.2f} {lons[i+1]:.7f},{lats[i+1]:.7f},{alts[i+1]:.2f}</coordinates>
  </LineString>
</Placemark>''')

kml.append(
     f''' <Placemark>
    <name>Predicted Observer Position</name>
    <styleUrl>#observerStyle</styleUrl>
    <Point>
      <altitudeMode>absolute</altitudeMode>
      <coordinates>{lon_avg:.8f},{lat_avg:.8f},{alt_avg_m:.2f}</coordinates>
    </Point>
  </Placemark> '''
)



kml.append('</Document></kml>')
with open('observer_line.kml', 'w') as f:
    f.write('\n'.join(kml))
from google.colab import files
files.download("observer_line.kml")

In [ ]:
#@title Visualization 2: Google Maps (requires API key)
!pip install gmplot -q
import gmplot
import portpicker
import threading
import http.server
import socketserver
from google.colab import output
from IPython.display import IFrame

# reuse the same line + altitude filtering logic as Section 8
t_samples_gmap = np.linspace(t_alt_min, t_alt_max, 200)
line_points_gmap = np.array([E0 + tt * direction for tt in t_samples_gmap])
line_latlonalt = np.array([
    pm.enu2geodetic(pt[0], pt[1], pt[2], lat0, lon0, alt0)
    for pt in line_points_gmap
])


if len(line_latlonalt) < 2:
    print("No (or only one) non-negative-altitude point on the line -- nothing to plot.\n"
          "Try widening f_uncertainty_frac in Section 6, or check your inputs.")
else:
    lats, lons, alts = line_latlonalt[:, 0], line_latlonalt[:, 1], line_latlonalt[:, 2]
    center_lat, center_lon = lats.mean(), lons.mean()

    gmap = gmplot.GoogleMapPlotter(center_lat, center_lon, 16, apikey=GOOGLE_MAPS_API_KEY)
    gmap.map_type = "satellite"  # or "roadmap", "hybrid", "terrain"

    # draw the line as many short colored segments, gray (0 alt) -> green (max alt on the line)
    gray = np.array([0x88, 0x88, 0x88])
    green = np.array([0x1a, 0x9e, 0x1a])
    red = np.array([0x9e, 0x1a, 0x1a])

    # Establish limits, avoiding division by zero
    max_alt = max(alts.max(), 1e-6)
    min_alt = min(alts.min(), -1e-6)  # Must be negative for the math to work

    for i in range(len(lats) - 1):
        # 1. Get the average altitude of the current segment
        avg_alt = (alts[i] + alts[i + 1]) / 2

        # 2. Determine color based on whether altitude is positive or negative
        if avg_alt >= 0:
            # Positive: 0 (gray) to max_alt (green)
            frac = avg_alt / max_alt
            rgb = (gray + frac * (green - gray)).astype(int)
        else:
            # Negative: min_alt (red) to 0 (gray)
            # Use avg_alt / min_alt so frac is 1.0 at absolute minimum and 0.0 at zero
            frac = avg_alt / min_alt
            rgb = (gray + frac * (red - gray)).astype(int)

        # 3. Convert to hex and plot
        color = '#%02x%02x%02x' % tuple(rgb)
        gmap.plot([lats[i], lats[i + 1]], [lons[i], lons[i + 1]], color=color, edge_width=6)


    # landmarks
    for i, (lat, lon, altv) in enumerate(landmarks_lla):
        gmap.marker(lat, lon, color='black', title=f"Landmark {i+1} (alt {altv:.0f} m)")

    # best-guess position (t=0)
    gmap.marker(obs_lat, obs_lon, color='red', title=f"Predicted Observer Position, alt {obs_alt:.0f} m")

    gmap.draw("observer_line_googlemaps.html")


port = portpicker.pick_unused_port()
def server_thread():
    class QuietHandler(http.server.SimpleHTTPRequestHandler):
        def log_message(self, format, *args): return
    handler = QuietHandler
    with socketserver.TCPServer(("", port), handler) as httpd:
        httpd.serve_forever()
threading.Thread(target=server_thread, daemon=True).start()
output.serve_kernel_port_as_iframe(port, path='/observer_line_googlemaps.html', width="100%", height="650")